# 01 · Whisper ASR: From Transcription to WER Evaluation

**Hardware**: 🟢 CPU works (defaults to whisper-small ~460MB; auto-switches to large-v3-turbo on GPU)

## What you will learn

1. Whisper's encoder-decoder architecture: mel spectrogram in, text tokens out
2. How transcription and timestamps are switched via **special-token prefixes** (Whisper's neatest design)
3. Evaluating with WER (word error rate) instead of "sounds about right"
4. The multi-x speedup of faster-whisper (CTranslate2 int8)

Theory reference: [theory.md](../theory.md) §2.

In [ ]:
%pip install -q torch transformers datasets soundfile jiwer faster-whisper accelerate

In [ ]:
import torch

if torch.cuda.is_available():
    device, model_id = "cuda:0", "openai/whisper-large-v3-turbo"
else:
    device, model_id = "cpu", "openai/whisper-small"
print(f"device={device}, model={model_id}")

## 1. Prepare test audio

We use LibriSpeech samples (they ship human reference transcripts, handy for WER later). Swap `wav` for your own recording anytime.

In [ ]:
from datasets import load_dataset
from IPython.display import Audio as AudioPlayer, display
import soundfile as sf

ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
sample = ds[0]
wav, sr = sample["audio"]["array"], sample["audio"]["sampling_rate"]
reference = sample["text"].lower()

sf.write("sample.wav", wav, sr)  # saved to disk for faster-whisper later
print(f"duration {len(wav)/sr:.1f}s | reference: {reference}")
display(AudioPlayer(wav, rate=sr))

## 2. Whisper transcription

Whisper encodes the task as **prefix tokens** for the decoder:

```
<|startoftranscript|> <|en|> <|transcribe|> <|notimestamps|> text...
```

Change language = change `<|en|>`; change task (translate to English) = swap `<|transcribe|>` for `<|translate|>`. One model, many tasks, all via the prompt — "prompt engineering" from 2022.

In [ ]:
from transformers import pipeline
import time

asr = pipeline("automatic-speech-recognition", model=model_id, device=device)

t0 = time.perf_counter()
result = asr({"array": wav, "sampling_rate": sr})
t_hf = time.perf_counter() - t0

print(f"[{t_hf:.2f}s] {result['text']}")

In [ ]:
# With timestamps (subtitle scenarios): Whisper predicts time tokens like <|0.00|>
result_ts = asr({"array": wav, "sampling_rate": sr}, return_timestamps=True)
for chunk in result_ts["chunks"]:
    print(f"{chunk['timestamp']}  {chunk['text']}")

## 3. Scientific evaluation: WER

WER = (substitutions + deletions + insertions) / reference words. Normalize text first (case, punctuation) — otherwise `Hello.` vs `hello` counts as an error.

In [ ]:
import jiwer

norm = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(),
])

# Average WER over the whole dummy set
refs, hyps = [], []
for s in ds:
    refs.append(norm(s["text"]))
    hyps.append(norm(asr({"array": s["audio"]["array"], "sampling_rate": sr})["text"]))

wer = jiwer.wer(refs, hyps)
print(f"WER on {len(ds)} samples: {wer:.2%}")
print("(LibriSpeech clean read speech is easy; accents/noise/jargon in the wild are the real test)")

## 4. faster-whisper: production-grade speed

Same weights, inference rewritten in CTranslate2 with int8 quantization — typically 3–5x faster on CPU. A classic "research model → production deployment" case study.

In [ ]:
from faster_whisper import WhisperModel

fw_size = "large-v3-turbo" if torch.cuda.is_available() else "small"
fw = WhisperModel(fw_size, device="cuda" if torch.cuda.is_available() else "cpu",
                  compute_type="float16" if torch.cuda.is_available() else "int8")

t0 = time.perf_counter()
segments, info = fw.transcribe("sample.wav")
text_fw = " ".join(s.text for s in segments)
t_fw = time.perf_counter() - t0

print(f"transformers: {t_hf:.2f}s")
print(f"faster-whisper: {t_fw:.2f}s")
print(f"detected language: {info.language} (p={info.language_probability:.2f})")
print(f"transcript: {text_fw}")

## Exercises

1. Record yourself (or grab a podcast clip) in another language; test `language="zh"` transcription and then `task="translate"` for direct English output.
2. Add Gaussian noise (SNR 10dB) and measure how much WER degrades — feel what "robustness comes from data" means.
3. Plot the WER/speed curve for `whisper-small` vs `whisper-large-v3-turbo` and decide how you'd choose in production.
4. Advanced: try NVIDIA Parakeet TDT (see [landscape.md](../landscape.md)) and compare latency in streaming scenarios.

**Next stop**: [02_kokoro_tts.ipynb](02_kokoro_tts.ipynb) — the reverse direction: text to speech.